# 01 - Làm sạch dữ liệu & tạo đặc trưng (Data Cleaning & Feature Engineering)

**Đề tài:** Ứng dụng học máy và GIS trong phân tích, phân loại mức độ ưu tiên quản lý rừng tại Nghệ An và Lâm Đồng

**Dữ liệu:** REDD+ — 19.545 bản ghi điểm (Lat/Long), thuộc 4 huyện: Kỳ Sơn, Tương Dương (Nghệ An); Lạc Dương, Di Linh (Lâm Đồng)

**Mục tiêu của notebook:**
1. Đọc dữ liệu thô, kiểm tra chất lượng (thiếu, trùng, kiểu dữ liệu, toạ độ).
2. Chuẩn hoá tên cột và giá trị text.
3. Xử lý trùng lặp, giá trị thiếu, ngày tháng.
4. Tạo đặc trưng có ý nghĩa lâm học từ `Land type` (nhóm hiện trạng rừng + cấp trữ lượng) và từ `Forest owner` (loại chủ rừng).
5. Xuất file sạch `data/processed/redd_plus_cleaned.csv` cho các notebook sau.

> Thứ tự chạy: `01_data_cleaning` → `02_eda` → `03_modeling` → `04_spatial_analysis_gis`

## 1. Thiết lập & đọc dữ liệu

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# Tự nhận thư mục gốc dự án (chạy được cả khi mở notebook trong thư mục notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "redd_plus_en.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
TABLE_DIR = ROOT / "outputs" / "tables"
for d in (PROCESSED_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()
cleaning_log = [("Dữ liệu gốc", len(df))]
print(f"Kích thước dữ liệu gốc: {df.shape[0]:,} dòng × {df.shape[1]} cột")
df.head()

Kích thước dữ liệu gốc: 19,545 dòng × 17 cột


,ID,Latitude,Longtitude,Province,District,Commune,Sub-regional,Plot,Lot,Site,Forest owner,Land type,Original resources of forest,Date,Priority,Contractor,Status
0,1.0,11.5169,108.272668,Lam Dong,Di Linh,Tam Bo,678,3.0,26.0,Tam Bo,Trần Ngọc Thế,Medium coniferous natural forest,Natural forest,1/1/2018 0:00,High,NaN,Verified
1,2.0,11.4464,108.062816,Lam Dong,Di Linh,Gung Re,695,5.0,1.0,Gung Re village,Forestry One Member Limited Liability Company Di Linh,Medium coniferous natural forest,Natural forest,1/1/2018 0:00,High,K' BRim (Gaoh),Verified
2,3.0,11.4462,108.062886,Lam Dong,Di Linh,Gung Re,695,5.0,3.0,Gung Re village,Forestry One Member Limited Liability Company Di Linh,Natural wood forest in the mountains with medium broad leaves and medium con...,Natural forest,1/1/2018 0:00,High,K' BRim (Gaoh),Verified
3,4.0,11.8010,107.937944,Lam Dong,Di Linh,Dinh Trang Thuong,607,3.0,8.0,Dinh Trang Thuong,Board of protection forest management Tan Thuong,Empty land with regenerated wood trees,Natural forest,1/13/2018 0:00,Low,NaN,Not Verified
4,5.0,11.4814,108.010331,Lam Dong,Di Linh,Lien Dam,660B,7.0,43.0,Lien Dam,Group Joint Stock Company Tan Mai,Other plantation forest,Plantation forest,1/13/2018 0:00,Low,NaN,Not Verified


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19545 entries, 0 to 19544
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ID                            19545 non-null  float64
 1   Latitude                      19545 non-null  float64
 2   Longtitude                    19545 non-null  float64
 3   Province                      19545 non-null  object 
 4   District                      19545 non-null  object 
 5   Commune                       19544 non-null  object 
 6   Sub-regional                  19544 non-null  object 
 7   Plot                          19304 non-null  float64
 8   Lot                           18456 non-null  float64
 9   Site                          19349 non-null  object 
 10  Forest owner                  14541 non-null  object 
 11  Land type                     19520 non-null  object 
 12  Original resources of forest  19545 non-null  object 
 13  D

## 2. Kiểm tra chất lượng dữ liệu ban đầu

In [3]:
missing = (df.isna().sum().to_frame("so_thieu")
           .assign(ty_le_pct=lambda t: (t["so_thieu"] / len(df) * 100).round(2))
           .sort_values("so_thieu", ascending=False))
missing

,so_thieu,ty_le_pct
Contractor,19301,98.75
Forest owner,5004,25.60
Lot,1089,5.57
Plot,241,1.23
Site,196,1.00
Land type,25,0.13
Date,1,0.01
Commune,1,0.01
Sub-regional,1,0.01
ID,0,0.00


In [4]:
print("Số ID trùng                         :", df["ID"].duplicated().sum())
print("Số dòng trùng hoàn toàn (kể cả ID)  :", df.duplicated().sum())
print("Số dòng trùng hoàn toàn (bỏ qua ID) :", df.drop(columns="ID").duplicated().sum())
print("Số dòng trùng toạ độ (Lat, Long)    :", df.duplicated(["Latitude", "Longtitude"]).sum())
print("\nID nhỏ nhất / lớn nhất:", df["ID"].min(), "/", df["ID"].max())

Số ID trùng                         : 1
Số dòng trùng hoàn toàn (kể cả ID)  : 1
Số dòng trùng hoàn toàn (bỏ qua ID) : 693
Số dòng trùng toạ độ (Lat, Long)    : 1777

ID nhỏ nhất / lớn nhất: 1.0 / 417000000000.0


**Nhận xét:**
- `Contractor` thiếu ~99% → loại bỏ.
- Có **693 dòng trùng hoàn toàn** (chỉ khác ID) — nếu chỉ kiểm tra trùng theo `ID` sẽ bỏ sót.
  Các dòng này giống hệt nhau về toạ độ, ngày, chủ rừng, loại đất, nhãn → là bản ghi nhập lặp, cần loại để tránh
  cùng một điểm vừa nằm trong train vừa nằm trong test (rò rỉ dữ liệu).
- Có ID bất thường rất lớn (~4,17×10¹¹) → lỗi nhập liệu, xem bên dưới.

## 3. Chuẩn hoá tên cột

Đổi sang dạng `snake_case` không dấu cách: dễ code hơn và **ArcGIS/Power BI không chấp nhận tên trường có dấu cách/ký tự đặc biệt**.
Đồng thời sửa lỗi chính tả `Longtitude` → `longitude`.

In [5]:
COLUMN_MAP = {
    "ID": "id", "Latitude": "latitude", "Longtitude": "longitude",
    "Province": "province", "District": "district", "Commune": "commune",
    "Sub-regional": "sub_region", "Plot": "plot", "Lot": "lot", "Site": "site",
    "Forest owner": "forest_owner", "Land type": "land_type",
    "Original resources of forest": "forest_origin", "Date": "date",
    "Priority": "priority", "Contractor": "contractor", "Status": "status",
}
df = df.rename(columns=COLUMN_MAP)
df.columns.tolist()

['id', 'latitude', 'longitude', 'province', 'district', 'commune', 'sub_region', 'plot', 'lot', 'site', 'forest_owner', 'land_type', 'forest_origin', 'date', 'priority', 'contractor', 'status']

## 4. Chuẩn hoá giá trị text
- Xoá khoảng trắng đầu/cuối, gộp nhiều khoảng trắng liên tiếp thành 1 (ví dụ `"Rich natural  evergreen..."`).
- Sửa lỗi lặp từ trong `land_type` (`"broadleaf broadleaf"`).
- Kiểm tra các cột phân loại chính có giá trị lạ/viết hoa thường không đồng nhất không.

In [6]:
text_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
for c in text_cols:
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

df["land_type"] = df["land_type"].str.replace(r"\b(\w+) \1\b", r"\1", regex=True)

for c in ["province", "district", "forest_origin", "priority", "status"]:
    print(f"{c:15s}:", sorted(df[c].dropna().unique()))

# Kiểm tra trùng tên khi bỏ khác biệt hoa/thường (ví dụ "Na Ngoi" vs "Na ngoi")
for c in ["commune", "site", "land_type"]:
    s = df[c].dropna()
    n_raw, n_lower = s.nunique(), s.str.lower().nunique()
    print(f"{c:10s}: {n_raw} giá trị, {n_lower} giá trị sau khi lower() -> chênh {n_raw - n_lower}")

province       : ['Lam Dong', 'Nghe An']
district       : ['Di Linh', 'Ky Son', 'Lac Duong', 'Tuong Duong']
forest_origin  : ['Natural forest', 'Plantation forest']
priority       : ['High', 'Low', 'Medium']
status         : ['Not Verified', 'Verified']
commune   : 35 giá trị, 35 giá trị sau khi lower() -> chênh 0
site      : 131 giá trị, 131 giá trị sau khi lower() -> chênh 0
land_type : 50 giá trị, 50 giá trị sau khi lower() -> chênh 0


In [7]:
# Nếu còn tên chỉ khác nhau hoa/thường thì đồng nhất theo cách viết phổ biến nhất
for c in ["commune", "site", "land_type"]:
    key = df[c].str.lower()
    canonical = df.groupby(key)[c].agg(lambda s: s.value_counts().index[0])
    df[c] = key.map(canonical).where(df[c].notna())
print("Số loại land_type sau chuẩn hoá:", df["land_type"].nunique())

Số loại land_type sau chuẩn hoá: 50


## 5. Xử lý ID và trùng lặp

In [8]:
print("Bản ghi có ID bất thường (> 1.000.000):")
display(df[df["id"] > 1e6][["id", "latitude", "longitude", "province", "district", "date", "priority"]])

Bản ghi có ID bất thường (> 1.000.000):


,id,latitude,longitude,province,district,date,priority
19007,4.170000e+11,19.27342,104.23463,Nghe An,Ky Son,NaN,Medium


In [9]:
n0 = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "id"], keep="first")
print(f"Đã loại {n0 - len(df)} dòng trùng hoàn toàn (bỏ qua ID)")
cleaning_log.append(("Sau khi bỏ dòng trùng hoàn toàn", len(df)))
print("Số ID còn trùng:", df["id"].duplicated().sum())

Đã loại 693 dòng trùng hoàn toàn (bỏ qua ID)
Số ID còn trùng: 0


## 6. Xử lý cột ngày tháng

Dữ liệu ở dạng `tháng/ngày/năm giờ:phút` (ví dụ `1/13/2018 0:00`) → chỉ định rõ `format` thay vì để pandas tự đoán
(tự đoán có thể đọc nhầm ngày/tháng với các giá trị ≤ 12).

In [10]:
df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y %H:%M", errors="coerce")
print("Số dòng không đọc được ngày:", df["date"].isna().sum())
display(df[df["date"].isna()][["id", "province", "district", "priority"]])

df = df.dropna(subset=["date"])            # 1 dòng: chính là dòng ID bất thường ở trên
cleaning_log.append(("Sau khi bỏ dòng không có ngày", len(df)))
df["id"] = df["id"].astype("int64")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
print("Khoảng thời gian:", df["date"].min().date(), "→", df["date"].max().date())
pd.crosstab(df["province"], df["year"])

Số dòng không đọc được ngày: 1


,id,province,district,priority
19007,4.170000e+11,Nghe An,Ky Son,Medium


year,2015,2016,2017,2018,2019,2020,2021,2022
province,,,,,,,,
Lam Dong,357,555,454,1654,2093,277,0,0
Nghe An,0,0,0,69,68,7141,3698,2485


**Lưu ý quan trọng:** Lâm Đồng chỉ có dữ liệu 2015–2020, Nghệ An chủ yếu 2020–2022 → `year` gần như trùng với
`province`. Vì vậy `year` **không nên dùng làm đặc trưng cho mô hình** (mô hình sẽ học "năm = tỉnh" chứ không học
được quy luật quản lý rừng, và không có ý nghĩa khi dự đoán lô rừng mới).

## 7. Kiểm tra toạ độ

Chỉ kiểm tra toạ độ nằm trong khung cả nước (8–24°N) là quá lỏng. Ở đây kiểm tra 4 mức:
1. Toạ độ hợp lệ & không bị đảo Lat/Long.
2. Nằm trong khung ranh giới **tỉnh của chính nó** (theo ranh giới hành chính *trước* sáp nhập 2025, vì dữ liệu ghi nhận 2015–2022).
3. Không nằm quá xa các điểm khác cùng huyện (phát hiện điểm "lạc huyện").
4. Độ chính xác (số chữ số thập phân) — 1 chữ số thập phân ≈ sai số 11 km, 2 chữ số ≈ 1,1 km.

Nếu có shapefile ranh giới huyện (ví dụ GADM level 2) đặt tại `data/external/`, có thể kiểm tra chính xác bằng
`geopandas.sjoin` (đoạn code tuỳ chọn ở cuối mục này).

In [11]:
# Khung bao (bounding box) gần đúng của 2 tỉnh theo ranh giới cũ (trước 07/2025)
PROVINCE_BBOX = {
    "Nghe An":  {"lat": (18.55, 20.05), "lon": (103.85, 105.85)},
    "Lam Dong": {"lat": (11.15, 12.45), "lon": (107.25, 108.80)},
}

def in_bbox(r):
    b = PROVINCE_BBOX[r["province"]]
    return b["lat"][0] <= r["latitude"] <= b["lat"][1] and b["lon"][0] <= r["longitude"] <= b["lon"][1]

df["in_province_bbox"] = df.apply(in_bbox, axis=1)
print("Toạ độ thiếu:", df[["latitude", "longitude"]].isna().any(axis=1).sum())
print("Điểm nằm ngoài khung tỉnh của chính nó:", (~df["in_province_bbox"]).sum())

df.groupby(["province", "district"])[["latitude", "longitude"]].agg(["min", "median", "max"]).round(3)

Toạ độ thiếu: 0
Điểm nằm ngoài khung tỉnh của chính nó: 0


latitude                 longitude                  
                          min  median     max       min   median      max
province district                                                        
Lam Dong Di Linh       11.239  11.423  11.804   107.905  108.056  108.299
         Lac Duong     11.929  12.066  12.309   108.284  108.513  108.717
Nghe An  Ky Son        19.199  19.302  19.409   104.007  104.174  104.286
         Tuong Duong   19.012  19.189  19.300   104.409  104.628  104.722

In [12]:
# Mức 3: khoảng cách (km) tới trung vị toạ độ của huyện, đánh dấu điểm xa bất thường
lat_med = df.groupby("district")["latitude"].transform("median")
lon_med = df.groupby("district")["longitude"].transform("median")
dy = (df["latitude"] - lat_med) * 111.0
dx = (df["longitude"] - lon_med) * 111.0 * np.cos(np.deg2rad(df["latitude"]))
df["dist_to_district_center_km"] = np.sqrt(dx ** 2 + dy ** 2).round(2)

q = df.groupby("district")["dist_to_district_center_km"].describe(percentiles=[0.5, 0.99]).round(1)
display(q)

# Ngưỡng: > 60 km so với tâm huyện là bất thường (mỗi huyện rộng khoảng 40-80 km)
df["coord_outlier"] = df["dist_to_district_center_km"] > 60
print("Số điểm xa tâm huyện > 60 km:", df["coord_outlier"].sum())

,count,mean,std,min,50%,99%,max
district,,,,,,,
Di Linh,413.0,13.9,8.9,0.4,11.4,43.8,44.9
Ky Son,8733.0,7.8,4.1,0.3,7.3,18.5,18.7
Lac Duong,4977.0,14.6,6.8,0.2,14.4,27.2,31.9
Tuong Duong,4728.0,7.0,3.9,0.4,6.0,17.3,28.5


In [13]:
# Mức 4: độ chính xác toạ độ = số chữ số thập phân của bản ghi gốc
def n_decimals(x):
    s = repr(float(x))
    return len(s.split(".")[1].rstrip("0")) if "." in s else 0

df["coord_decimals"] = np.minimum(df["latitude"].map(n_decimals), df["longitude"].map(n_decimals))
display(pd.crosstab(df["coord_decimals"], df["province"]))

df["low_precision_coord"] = df["coord_decimals"] <= 2
print("Số điểm có toạ độ ≤ 2 chữ số thập phân (sai số có thể ≥ 1 km):", df["low_precision_coord"].sum())

province,Lam Dong,Nghe An
coord_decimals,,
0,1,0
1,0,47
2,5,604
3,47,4447
4,485,1326
5,839,2009
6,4013,4952
7,0,76


In [14]:
# (Tuỳ chọn) Kiểm tra chính xác bằng shapefile ranh giới huyện nếu có GeoPandas + file ranh giới
BOUNDARY = ROOT / "data" / "external" / "gadm41_VNM_2.json"
try:
    import geopandas as gpd
    if BOUNDARY.exists():
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326")
        districts = gpd.read_file(BOUNDARY)[["NAME_1", "NAME_2", "geometry"]]
        joined = gpd.sjoin(gdf, districts, how="left", predicate="within")
        print("Điểm không rơi vào huyện nào:", joined["NAME_2"].isna().sum())
        display(pd.crosstab(joined["district"], joined["NAME_2"].fillna("(ngoài ranh giới)")))
    else:
        print(f"Chưa có file ranh giới {BOUNDARY.name} → bỏ qua bước sjoin (đã kiểm tra bằng bbox ở trên).")
except ImportError:
    print("Chưa cài geopandas → bỏ qua bước sjoin (đã kiểm tra bằng bbox ở trên).")

Chưa cài geopandas → bỏ qua bước sjoin (đã kiểm tra bằng bbox ở trên).


**Quyết định xử lý toạ độ:**
- Loại các điểm nằm ngoài khung tỉnh hoặc xa tâm huyện bất thường (nếu có).
- Điểm toạ độ độ chính xác thấp: **giữ lại** cho mô hình (nhãn và thuộc tính vẫn đúng), nhưng gắn cờ `low_precision_coord`
  để khi vẽ bản đồ/phân tích không gian có thể lọc ra và nêu trong phần hạn chế.
- Điểm trùng toạ độ nhưng khác ngày/khác nhãn (sau khi đã bỏ trùng hoàn toàn) vẫn giữ — đó có thể là các lần ghi nhận khác nhau
  tại cùng một vị trí. Đếm số lần trùng vào cột `n_same_coord` để tham khảo.
- Các vị trí **trùng toạ độ nhưng khác nhãn `priority`** là dấu hiệu **nhiễu nhãn** (cùng một lô nhưng mỗi lần ghi nhận
  được xếp mức khác nhau). Không thể biết nhãn nào đúng nên vẫn giữ, nhưng điều này đặt ra **giới hạn trên** cho độ chính xác
  của mọi mô hình (phân tích thêm ở notebook 02, mục 8).

In [15]:
n0 = len(df)
df = df[df["in_province_bbox"] & ~df["coord_outlier"]].copy()
print(f"Đã loại {n0 - len(df)} điểm có toạ độ sai/ngoài ranh giới")
cleaning_log.append(("Sau khi bỏ toạ độ sai", len(df)))

df["n_same_coord"] = df.groupby(["latitude", "longitude"])["id"].transform("size")
print("Số điểm có ≥ 2 bản ghi tại cùng toạ độ:", (df["n_same_coord"] > 1).sum())
conflict = df.groupby(["latitude", "longitude"])["priority"].nunique()
print("Số vị trí có nhiều nhãn Priority khác nhau:", (conflict > 1).sum())
n_rows_conflict = df.groupby(["latitude", "longitude"])["priority"].transform("nunique").gt(1).sum()
print("Số bản ghi nằm tại các vị trí mâu thuẫn nhãn:", int(n_rows_conflict))

Đã loại 0 điểm có toạ độ sai/ngoài ranh giới
Số điểm có ≥ 2 bản ghi tại cùng toạ độ: 1924
Số vị trí có nhiều nhãn Priority khác nhau: 312
Số bản ghi nằm tại các vị trí mâu thuẫn nhãn: 763


## 8. Xử lý giá trị thiếu

| Cột | % thiếu | Cách xử lý | Lý do |
|---|---|---|---|
| `contractor` | ~99% | Loại bỏ cột | Gần như không có dữ liệu |
| `forest_owner` | ~26% | Điền `"Unknown"` | "Không rõ chủ" là một thông tin (đa số là đất chưa giao), giữ thành 1 nhóm riêng |
| `land_type` | ~0,1% | Điền `"Unknown"` | Không điền bằng mode để tránh tạo thông tin giả |
| `commune`, `sub_region`, `site` | rất ít | Điền `"Unknown"` | Không bỏ dòng để giữ dữ liệu toạ độ |
| `plot`, `lot` | ít | Giữ trống (NA) | Chỉ là mã định danh, không dùng cho mô hình |

In [16]:
df = df.drop(columns=["contractor"])
for c in ["forest_owner", "land_type", "commune", "sub_region", "site"]:
    df[c] = df[c].fillna("Unknown")
df["plot"] = df["plot"].astype("Int64")
df["lot"] = df["lot"].astype("Int64")

print("Giá trị thiếu còn lại:")
print(df.isna().sum()[df.isna().sum() > 0])

Giá trị thiếu còn lại:
plot     240
lot     1061
dtype: int64


## 9. Tạo đặc trưng từ `land_type`

`land_type` có 50 mô tả tự do. Nếu chỉ gom bằng vài từ khoá đơn giản (ví dụ tìm chữ "plantation") sẽ sai rất nhiều:
phần lớn lô thuộc nguồn gốc rừng trồng lại được mô tả là *"Area of mountains and land"*, *"Regenerating..."*, *"Area planted..."*
nên bị dồn nhầm vào nhóm rừng tự nhiên. Vì vậy ở đây kết hợp **cả `forest_origin` và `land_type`**, rồi tách `land_type` thành **2 đặc trưng có ý nghĩa lâm học** (theo hệ thống phân loại hiện trạng rừng Việt Nam,
Thông tư 33/2018/TT-BNNPTNT):

1. **`land_cover_group`** — nhóm hiện trạng:
   - Rừng tự nhiên: *Evergreen broadleaf* (lá rộng thường xanh), *Deciduous broadleaf* (rụng lá), *Coniferous* (lá kim),
     *Broadleaf-coniferous mixed* (hỗn giao lá rộng–lá kim), *Mixed* (hỗn giao, không ghi rõ), *Wood-bamboo mixed* (hỗn giao gỗ–tre nứa), *Bamboo* (tre nứa)
   - Không phải rừng tự nhiên: *Plantation forest* (rừng trồng), *Newly planted land* (đất đã trồng, chưa thành rừng),
     *Regenerating land* (đất có cây gỗ tái sinh), *Bare/rocky land* (đất trống, núi đá)
2. **`stock_level`** — cấp trữ lượng (chỉ với rừng gỗ tự nhiên), có **thứ bậc**:
   Rich (giàu, kể cả nguyên sinh) > Medium (trung bình) > Poor (nghèo) > Very poor (rất nghèo) ≈ Recovered (phục hồi) > Not applicable

In [17]:
def land_cover_group(land_type: str, origin: str) -> str:
    t = land_type.lower()
    if t == "unknown":
        return "Unknown"
    if origin == "Plantation forest":
        if "regenerat" in t:
            return "Regenerating land"
        if "rocky" in t or t == "area of mountains and land":
            return "Bare/rocky land"
        if "area planted" in t or "soil planted" in t:
            return "Newly planted land"
        return "Plantation forest"
    # Rừng tự nhiên
    if "bamboo" in t or "bambusa" in t:
        return "Wood-bamboo mixed" if "mixed" in t else "Bamboo"
    if "regenerat" in t or "empty land" in t:
        return "Regenerating land"
    if "coniferous" in t and "broad" in t:
        return "Broadleaf-coniferous mixed"
    if "coniferous" in t:
        return "Coniferous"
    if "deciduous" in t:
        return "Deciduous broadleaf"
    if "broad" in t:
        return "Evergreen broadleaf"
    if "mixed" in t:
        return "Mixed"
    return "Unknown"


STOCK_ORDER = {"Not applicable": 0, "Recovered": 1, "Very poor": 1, "Poor": 2, "Medium": 3, "Rich": 4}

def stock_level(land_type: str, group: str) -> str:
    t = land_type.lower()
    natural_wood = group in {"Evergreen broadleaf", "Deciduous broadleaf", "Coniferous",
                             "Broadleaf-coniferous mixed", "Mixed"}
    if not natural_wood:
        return "Not applicable"
    if "primeval" in t or "rich" in t:
        return "Rich"
    if "very poor" in t:
        return "Very poor"
    if "medium" in t:
        return "Medium"
    if "poor" in t:
        return "Poor"
    if "recover" in t or "restored" in t:
        return "Recovered"
    return "Not applicable"


df["land_cover_group"] = [land_cover_group(t, o) for t, o in zip(df["land_type"], df["forest_origin"])]
df["stock_level"] = [stock_level(t, g) for t, g in zip(df["land_type"], df["land_cover_group"])]
df["stock_ord"] = df["stock_level"].map(STOCK_ORDER)

# Bảng tra để kiểm tra lại từng giá trị land_type đã được gom đúng chưa (nên đưa vào phụ lục báo cáo)
mapping_table = (df.groupby(["forest_origin", "land_type", "land_cover_group", "stock_level"])
                   .size().rename("so_ban_ghi").reset_index()
                   .sort_values(["forest_origin", "land_cover_group", "so_ban_ghi"], ascending=[True, True, False]))
mapping_table.to_csv(TABLE_DIR / "land_type_mapping.csv", index=False, encoding="utf-8-sig")
mapping_table

,forest_origin,land_type,land_cover_group,stock_level,so_ban_ghi
10,Natural forest,Natural bamboo forest on the soil mountain,Bamboo,Not applicable,152
11,Natural forest,Natural bambusa procera forest,Bamboo,Not applicable,27
24,Natural forest,Other natural bamboo forest on the soil mountain,Bamboo,Not applicable,18
9,Natural forest,Natural bamboo forest,Bamboo,Not applicable,8
12,Natural forest,Natural bambusa procera forest on the soil mountain,Bamboo,Not applicable,3
5,Natural forest,Medium natural broad-leaved and coniferous forest on the soil mountain,Broadleaf-coniferous mixed,Medium,70
16,Natural forest,Natural wood forest in the mountains with broad leaves and rich coniferous l...,Broadleaf-coniferous mixed,Rich,62
15,Natural forest,Natural wood and poor broadleaf and coniferous forest on soil mountain,Broadleaf-coniferous mixed,Poor,51
14,Natural forest,Natural poor broadleaf and poor coniferous forests,Broadleaf-coniferous mixed,Poor,9
13,Natural forest,Natural medium broadleaf and medium coniferous forests,Broadleaf-coniferous mixed,Medium,3


In [18]:
display(pd.crosstab(df["land_cover_group"], df["forest_origin"], margins=True))
display(pd.crosstab(df["stock_level"], df["priority"], margins=True))

forest_origin,Natural forest,Plantation forest,All
land_cover_group,,,
Bamboo,208,0,208
Bare/rocky land,0,3514,3514
Broadleaf-coniferous mixed,196,0,196
Coniferous,3548,0,3548
Deciduous broadleaf,16,0,16
Evergreen broadleaf,4025,0,4025
Mixed,2915,0,2915
Newly planted land,0,914,914
Plantation forest,0,1176,1176


priority,High,Low,Medium,All
stock_level,,,,
Medium,846,15,1502,2363
Not applicable,147,6787,1217,8151
Poor,1019,31,2030,3080
Recovered,253,1,1326,1580
Rich,895,0,2765,3660
Very poor,2,1,14,17
All,3162,6835,8854,18851


## 10. Tạo đặc trưng loại chủ rừng từ `forest_owner`

`forest_owner` có ~490 giá trị (đa số là tên hộ gia đình) → không dùng trực tiếp được. Gom thành **loại chủ rừng**:
Ban quản lý rừng/VQG, UBND xã, Doanh nghiệp, Tổ chức/đoàn thể – cộng đồng, Hộ gia đình/cá nhân, Không rõ.

In [19]:
def owner_type(owner: str) -> str:
    o = owner.lower()
    if o == "unknown":
        return "Unknown"
    if "board" in o or "bql" in o or "national park" in o or "management" in o:
        return "Management board"
    if "committee" in o:
        return "Commune committee"
    if "company" in o or "enterprise" in o or "corporation" in o:
        return "Company"
    if "youth" in o or "team" in o or "community" in o or "village" in o or "group" in o:
        return "Organization/community"
    return "Household/individual"        # tên người, "HGD" (hộ gia đình), ...

df["owner_type"] = df["forest_owner"].map(owner_type)
pd.crosstab(df["owner_type"], df["priority"], margins=True)

priority,High,Low,Medium,All
owner_type,,,,
Commune committee,177,1294,1315,2786
Company,77,150,184,411
Household/individual,192,591,1035,1818
Management board,626,3583,3361,7570
Organization/community,66,723,474,1263
Unknown,2024,494,2485,5003
All,3162,6835,8854,18851


## 11. Khoá nhóm không gian cho đánh giá mô hình

`sub_region` (tiểu khu) là đơn vị quản lý không gian. Các điểm trong cùng tiểu khu rất giống nhau → khi đánh giá mô hình
sẽ chia train/test **theo tiểu khu** (GroupKFold) để tránh rò rỉ không gian. Một số mã tiểu khu trùng nhau giữa 2 tỉnh
nên ghép thêm tên tỉnh để tạo khoá duy nhất.

In [20]:
df["spatial_group"] = df["province"].str.replace(" ", "") + "_" + df["sub_region"].astype(str)
print("Số tiểu khu (nhóm không gian):", df["spatial_group"].nunique())
print(df.groupby("province")["spatial_group"].nunique())

Số tiểu khu (nhóm không gian): 303
province
Lam Dong    205
Nghe An      98
Name: spatial_group, dtype: int64


## 12. Lưu dữ liệu đã làm sạch

In [21]:
# Bỏ các cột cờ trung gian không cần nữa
df = df.drop(columns=["in_province_bbox", "coord_outlier"])

col_order = ["id", "latitude", "longitude", "province", "district", "commune", "sub_region", "spatial_group",
             "plot", "lot", "site", "forest_owner", "owner_type", "land_type", "forest_origin",
             "land_cover_group", "stock_level", "stock_ord", "date", "year", "month", "status", "priority",
             "dist_to_district_center_km", "coord_decimals", "low_precision_coord", "n_same_coord"]
df = df[col_order].reset_index(drop=True)

out_path = PROCESSED_DIR / "redd_plus_cleaned.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")   # utf-8-sig: Excel/Power BI đọc đúng tiếng Việt
print(f"Đã lưu: {out_path.relative_to(ROOT)}  —  {df.shape[0]:,} dòng × {df.shape[1]} cột")

Đã lưu: data\processed\redd_plus_cleaned.csv  —  18,851 dòng × 27 cột


In [22]:
log_df = pd.DataFrame(cleaning_log, columns=["buoc", "so_dong"])
log_df["so_dong_bi_loai"] = (-log_df["so_dong"].diff()).fillna(0).astype(int)
log_df.to_csv(TABLE_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
print(log_df.to_string(index=False))

print("\n--- Phân bố Priority ---")
print(df["priority"].value_counts())
print("\n--- Phân bố Status ---")
print(df["status"].value_counts(normalize=True).round(4))

                           buoc  so_dong  so_dong_bi_loai
                    Dữ liệu gốc    19545                0
Sau khi bỏ dòng trùng hoàn toàn    18852              693
  Sau khi bỏ dòng không có ngày    18851                1
          Sau khi bỏ toạ độ sai    18851                0

--- Phân bố Priority ---
priority
Medium    8854
Low       6835
High      3162
Name: count, dtype: int64

--- Phân bố Status ---
status
Not Verified    0.9875
Verified        0.0125
Name: proportion, dtype: float64


## Kết luận bước 1

- Từ 19.545 dòng gốc, loại **693 dòng trùng hoàn toàn** và 1 dòng lỗi (ID bất thường, không có ngày) → còn ~18.850 dòng.
- Toạ độ đều nằm đúng ranh giới tỉnh/huyện; một phần điểm ở Nghệ An có độ chính xác thấp (≤ 2 chữ số thập phân) → gắn cờ.
- Tạo các đặc trưng mới có ý nghĩa: `land_cover_group` (12 nhóm hiện trạng), `stock_level`/`stock_ord` (cấp trữ lượng có thứ bậc),
  `owner_type` (loại chủ rừng), `spatial_group` (tiểu khu, dùng cho GroupKFold).
- `status` lệch rất nặng (Verified ~1,2%) → không phù hợp làm biến mục tiêu; chọn `priority` là hợp lý.
- `year` trùng gần hết với tỉnh → không dùng làm đặc trưng mô hình.
- Có hàng trăm vị trí trùng toạ độ nhưng mang nhãn `priority` khác nhau → dữ liệu có **nhiễu nhãn**, cần lưu ý khi đánh giá
  mô hình (không kỳ vọng độ chính xác gần 100%).